#### Loading the data

In [ ]:
!pip install pypdf

In [ ]:
from langchain.document_loaders.pdf import PyPDFDirectoryLoader

documents=PyPDFDirectoryLoader("data").load()

In [ ]:
# checking the documents
documents

In [ ]:
# GLOBAL VARIABLES
CHROMA_PATH='chroma'
DATA="data"

### Spliting data from documents(one page) into smaller chunks

In [ ]:
!pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.document import Document

def split_docs(documents:list[Document]):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False
    )

    return text_splitter.split_documents(documents)

In [ ]:
chunks=split_docs(documents)

In [ ]:
chunks

In [ ]:
def calculate_chunk_id(chunks):

    last_page_id=None
    curr_chunk_index=0

    for chunk in chunks:
        source=chunk.metadata.get("source")
        page=chunk.metadata.get("page")
        curr_page_id=f"{source}:{page}"
        
        # checking for new page before alloting final chunk id
        if curr_page_id==last_page_id:
            curr_chunk_index+=1
        else:
            curr_chunk_index=0
        
        # denoting chunk ids
        chunk_id=f"{curr_page_id}:{curr_chunk_index}"

        # incrementing page id
        last_page_id = curr_page_id

        # updating metadata for chunks
        chunk.metadata["chunk_id"]=chunk_id

    return chunks

In [ ]:
# trying chunks id function just created

trial_chunks=calculate_chunk_id(chunks)

In [ ]:
for i in range(20):
    print(trial_chunks[i].metadata["chunk_id"])

In [ ]:
# creating embeddings from the text

# from langchain_community.embeddings.bedrock import BedrockEmbeddings 
# to be replaced with ollama embedings and nomic embed later for local run
from langchain_community.embeddings.ollama import OllamaEmbeddings
# embeddings = OllamaEmbeddings(model="nomic-embed-text")
# also run ollama pull nomic-embed-text for it to work, dont know how to run this in deployment phase
def get_embeddings():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    return embeddings

#### adding to chroma DB

In [ ]:
from langchain_chroma import Chroma

def emed_to_chromaDB(chunks: list[Document]):
    db=Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=get_embeddings()
    )
    # Calculate Page IDs.
    chunks_with_ids = calculate_chunk_id(chunks)
    
    # existing items
    existing_items=db.get(include=[])
    existing_ids=set(existing_items["ids"])

    print(f"existing no of docs in DB:{len(existing_ids)}")

    # only adding the documents that dont exist in the DB
    new_chunks=[]
    for chunk in chunks_with_ids:
        if chunk.metadata['chunk_id'] not in existing_ids:
            new_chunks.append(chunk)
    
    # adding the new chunks to the DB
    if len(new_chunks):
        print(f" Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["chunk_id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)

    else:
        print(" No new documents to add")


In [ ]:
# trying to add new documents to embeddings

emed_to_chromaDB(chunks)

## includng query in rag

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama

prompt_temp=""" 
    Answer the question with the folloeing context only
    {context}

    ___
    answer the question carefully with the above context {ques}
 """

def query_process(query_text: str):
    embedding_func=get_embeddings()
    db=Chroma(
        persist_directory=CHROMA_PATH,
        embedding_function=get_embeddings()
    )

    results=db.similarity_search_with_score(query_text,k=5)
    context_text="\n\n --- \n\n".join([doc.page_content for doc ,_score in results])
    prompt_temp_str=ChatPromptTemplate.from_template(prompt_temp)
    prompt=prompt_temp_str.format(context=context_text, ques=query_text)

    # printing the prompt
    model = Ollama(model="llama3.1:8b")
    response_text = model.invoke(prompt)
    source_for_answer = [doc.metadata.get("chunk_id") for doc, _score in results]
    final_answer=f"{response_text}. I got the answers from the following sources {source_for_answer}"
    print(final_answer)
    return final_answer


In [ ]:
# calling the query function for trials
query_process("is bhagavat gita a religious book?")